## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score, make_scorer
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

## 1. Preprocessing pipeline

In [2]:
df = pd.read_csv("../../datasets/checker_submits.csv")

class FeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):

        return self

    def transform(self, X, y=None):

        X_transformed = X.copy()

        X_transformed['timestamp'] = pd.to_datetime(X_transformed['timestamp'])

        X_transformed['hour'] = X_transformed['timestamp'].dt.hour

        X_transformed['weekday'] = X_transformed['timestamp'].dt.weekday

        X_transformed = X_transformed.drop(columns=['timestamp'])

        return X_transformed

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        self.categorical_columns_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.categorical_columns_:
            self.encoder_ = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            self.encoder_.fit(X[self.categorical_columns_])
        else:
            self.encoder_ = None
        return self

    def transform(self, X):
        if not self.categorical_columns_:
            return X.copy()

        encoded_array = self.encoder_.transform(X[self.categorical_columns_])
        feature_names = self.encoder_.get_feature_names_out(self.categorical_columns_)
        encoded_df = pd.DataFrame(encoded_array, columns=feature_names, index=X.index)

        numeric_df = X.drop(columns=self.categorical_columns_)
        return pd.concat([numeric_df, encoded_df], axis=1)

In [4]:
class TrainValidationTest:

    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state
        self.val_size = 0.25 

    def split(self, X, y):
 
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y  
        )

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_val, y_train_val,
            test_size=self.val_size,
            random_state=self.random_state,
            stratify=y_train_val
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

In [5]:
class ModelSelection:

    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = []  
        self.best_model_name = None  
        self.best_model = None
        self.best_valid_score = 0
        
    def choose(self, X_train, y_train, X_valid, y_valid):

        self.best_models = []
        self.best_model_name = None
        self.best_model = None
        self.best_valid_score = 0

        for idx, gs in enumerate(self.grids):
            model_name = self.grid_dict[idx]

            print(f"Estimator: {model_name}")

            with tqdm(total=1, desc=f"Training {model_name}") as pbar:
                gs.fit(X_train, y_train)
                pbar.update(1)

            best_estimator = gs.best_estimator_
            best_params = gs.best_params_
            best_train_score = gs.best_score_

            valid_score = best_estimator.score(X_valid, y_valid)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")

            model_info = {
                'model': model_name,
                'estimator': best_estimator,
                'params': best_params,
                'train_score': best_train_score,
                'valid_score': valid_score,
                'gs_object': gs
            }
            self.best_models.append(model_info)

            if valid_score > self.best_valid_score:
                self.best_valid_score = valid_score
                self.best_model_name = model_name
                self.best_model = best_estimator

        print(f"Classifier with best validation set accuracy: {self.best_model_name}")

        
        return self.best_model_name
    
    def best_results(self):

        results = []
        
        for model_info in self.best_models:
            results.append({
                'model': model_info['model'],
                'params': model_info['params'],
                'valid_score': model_info['valid_score']
            })
        
        return display(pd.DataFrame(results))

## 3. Finalization

In [6]:
class Finalize:

    def __init__(self, estimator):
        self.estimator = estimator
        self.final_accuracy = None
        
    def final_score(self, X_train, y_train, X_test, y_test):

        self.estimator.fit(X_train, y_train)

        y_pred = self.estimator.predict(X_test)

        self.final_accuracy = accuracy_score(y_test, y_pred)

        print(f"Accuracy of the final model is {self.final_accuracy}")
        
        return self.final_accuracy
    
    def save_model(self, path):

        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")


## 4. Main program

In [7]:
df = pd.read_csv("../../datasets/checker_submits.csv")

In [8]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder())
])

data = preprocessing.fit_transform(df)

In [9]:
y = data['weekday']
X = data.drop(columns=['weekday'])

splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)

In [10]:
svm = svm.SVC()
svm_params = [
    {
        'kernel': ('linear', 'rbf'), 
        'C': [0.01, 0.1, 1], 
        'gamma': ['scale'], 
        'class_weight': ('balanced', None), 
        'random_state': [21], 
        'probability': [True]
    }
]

gs_svm = GridSearchCV(
    estimator=svm, 
    param_grid=svm_params, 
    scoring='accuracy', 
    cv=2, 
    n_jobs=-1,
)

dt = DecisionTreeClassifier()
dt_params = [
    {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 10],
        'class_weight': ['balanced', None],
        'random_state': [21]
    }
]

gs_tree = GridSearchCV(
    estimator=dt,
    param_grid=dt_params,
    scoring='accuracy',
    cv=2,
    n_jobs=-1,
)

rf = RandomForestClassifier()
rf_params = [
    {
        'n_estimators': [50, 100],
        'criterion': ['gini', 'entropy'],
        'max_depth': [10, 20, 30],
        'min_samples_split': [5],
        'class_weight': ['balanced', None],
        'random_state': [21]
    }
]

gs_rf = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    scoring='accuracy',
    cv=2,
    n_jobs=-1,
)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

selector = ModelSelection(grids, grid_dict)

best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)

Estimator: SVM


Training SVM:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'C': 1, 'class_weight': None, 'gamma': 'scale', 'kernel': 'linear', 'probability': True, 'random_state': 21}
Best training accuracy: 0.623
Validation set accuracy score for best params: 0.623
Estimator: Decision Tree


Training Decision Tree:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.864
Estimator: Random Forest


Training Random Forest:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 30, 'min_samples_split': 5, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 0.833
Validation set accuracy score for best params: 0.861
Classifier with best validation set accuracy: Decision Tree


In [11]:
final = Finalize(selector.best_model)
final.final_score(X_train, y_train, X_test, y_test)
final.save_model(path = 'Decision Tree{0.86094}.sav')

Accuracy of the final model is 0.8609467455621301
Model was successfully saved to Decision Tree{0.86094}.sav


In [12]:
loaded_model = joblib.load('Decision Tree{0.86094}.sav')
final = Finalize(loaded_model)
final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.8609467455621301


0.8609467455621301